In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore, ScaleInvariantSignalDistortionRatio


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import * # FullSubPathExtension, FullSubPathExtension_3_heads, FullSubPathExtension_ver2, FullSubPathExtension_abs_pha, FullSubPathExtension_abs_pha_mapping, FullSubPathExtension_ver2_abs_pha, FullSubPathExtension_ver3

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3, model_eval_3_heads, use_pcs, inv_pcs, model_eval_old

import matplotlib.pyplot as plt

In [2]:
TEST_DIR = os.path.join("data", "DS_10283_2791", "clean_testset_wav")
TEST_NOISE_DIR = os.path.join("data", "DS_10283_2791", "noisy_testset_wav")
NOISE_DIR = os.path.join("data", "demand_test")

CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
import random

SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

In [4]:
# N_FFTS = 512
# HOP_LENGTH = 256
# HID_SIZE = 32
# SR = 16_000

HID_SIZE = 64
SR = 48_000# configs.sample_rate
BATCH_SIZE = 8 # 32

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [5]:
rir_dict = {1: os.path.join("data", "rirs48_small_3_test"), 1: os.path.join("data", "rirs48_medium_3_test"), 1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
dataset = TRUNetDataset(TEST_DIR, sr=SR, noise_dir=NOISE_DIR, rir_dir=rir_dict, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False)
dataset.set_epoch(99)

180
12


In [6]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [7]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [8]:
def pad_sequence(batch):
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    input_signal, target_signal, noise, rir = zip(*batch)
        
    max_len_s = max(s.shape[-1] for s in input_signal)
    
    padded_input = torch.zeros(len(input_signal), max_len_s)
    padded_target = torch.zeros(len(target_signal), max_len_s)
    
    for i, s in enumerate(input_signal):
        padded_input[i, :s.shape[-1]] = s
        padded_target[i, :s.shape[-1]] = target_signal[i]

    return padded_input, padded_target


def collate_fn(batch):
    
    padded_input, padded_target = pad_sequence(batch)
        
    padded_input = padded_input.reshape(-1, padded_input.shape[-1])
    padded_target = padded_target.reshape(-1, padded_input.shape[-1])

    return padded_input, padded_target

In [9]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [10]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torch_stoi import NegSTOILoss

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to("cuda")
stoi = NegSTOILoss(16_000, use_vad=False, do_resample=False).to("cuda")
sisdr = ScaleInvariantSignalDistortionRatio().to("cuda")
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [ ]:
from torchaudio.transforms import Resample
from thop import profile


def get_metrics(loader, device="cpu"):
    
    nisqa_scores_input = []
    pesq_scores_input = []
    stoi_scores_input = []
    sisdr_scores_input = []
    srmr_scores_input = []
    dnsmos_scores_input = []

    nisqa_scores_target = []
    pesq_scores_target = []
    stoi_scores_target = []
    sisdr_scores_target = []
    srmr_scores_target = []
    dnsmos_scores_target = []

    with torch.no_grad():
        for signal, target in tqdm(loader):
            signal = signal.to(device)
            target = target.to(device)
            
            min_l = min(signal.shape[-1], target.shape[-1])
            nisqa_score_input, _, _ = process(signal.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            nisqa_score_target, _, _ = process(target.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

            signal = signal[:, :min_l]
            target = target[:, :min_l]
            
            resampler = Resample(SR, 16_000)
            signal = resampler(signal.cpu()).cuda()
            target = resampler(target.cpu()).cuda()
            # min_l = min(output.shape[-1], target.shape[-1])

            stoi_score_target = stoi(target[..., :min_l], target[..., :min_l])
            stoi_score_input = stoi(signal[..., :min_l], target[..., :min_l])

            srmr_score_input = srmr(signal.detach().cpu())
            srmr_score_target = srmr(target.detach().cpu())

            sisdr_score_input = sisdr(signal, target)
            sisdr_score_target = sisdr(target, target)

            dnsmos_score_input = dnsmos(signal.detach())
            dnsmos_score_target = dnsmos(target.detach())

            try:
                pesq_score_input = pesq(signal[..., :min_l], target[..., :min_l])
                pesq_score_target = pesq(target[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores_input.append(nisqa_score_input[0])
            srmr_scores_input.append(srmr_score_input)
            stoi_scores_input.append(stoi_score_input.cpu())
            sisdr_scores_input.append(sisdr_score_input.cpu())
            pesq_scores_input.append(pesq_score_input.cpu())
            dnsmos_scores_input.append(dnsmos_score_input.cpu())

            nisqa_scores_target.append(nisqa_score_target[0])
            srmr_scores_target.append(srmr_score_target)
            stoi_scores_target.append(stoi_score_target.cpu())
            sisdr_scores_target.append(sisdr_score_target.cpu())
            pesq_scores_target.append(pesq_score_target.cpu())
            dnsmos_scores_target.append(dnsmos_score_target.cpu())

    result = {"nisqa": nisqa_scores_input, "stoi": stoi_scores_input, "sisdr": sisdr_scores_input, "srmr": srmr_scores_input, "pesq": pesq_scores_input, "dnsmos": dnsmos_scores_input}
    result_target = {"nisqa": nisqa_scores_target, "stoi": stoi_scores_target, "sisdr": sisdr_scores_target, "srmr": srmr_scores_target, "pesq": pesq_scores_target, "dnsmos": dnsmos_scores_target}
        
    return result, result_target

In [15]:
metrics, metrics_target = get_metrics(test_dataloader, device="cuda")

100%|██████████| 824/824 [22:34<00:00,  1.64s/it]


In [16]:
print("NISQA:", torch.vstack(metrics["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics["stoi"]).mean(dim=0))
print("SI-SDR:", -torch.vstack(metrics["sisdr"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics["dnsmos"]).mean(dim=0))
# print("MACs:", sum(metrics["macs"]) / len(metrics["macs"]))

NISQA: tensor([2.663, 2.395, 3.452, 3.488, 3.300])
PESQ: tensor([1.600])
SRMR: tensor([6.222])
STOI: tensor([1.])
SI-SDR: tensor([12.188])
DNSMOS: tensor([2.972, 3.045, 2.473, 2.243], dtype=torch.float64)


In [17]:
print("NISQA:", torch.vstack(metrics_target["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics_target["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics_target["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics_target["stoi"]).mean(dim=0))
print("SI-SDR:", -torch.vstack(metrics_target["sisdr"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics_target["dnsmos"]).mean(dim=0))
# print("MACs:", sum(metrics["macs"]) / len(metrics["macs"]))

NISQA: tensor([4.071, 4.118, 4.011, 4.141, 4.133])
PESQ: tensor([4.644])
SRMR: tensor([8.943])
STOI: tensor([0.855])
SI-SDR: tensor([-97.193])
DNSMOS: tensor([3.557, 3.464, 3.963, 3.153], dtype=torch.float64)


NISQA: tensor([3.781, 4.034, 3.757, 3.831, 3.922])
PESQ: tensor([2.375])
SRMR: tensor([9.390])
STOI: tensor([0.887])
SI-SDR: tensor([13.084])
DNSMOS: tensor([3.183, 3.034, 3.857, 2.731], dtype=torch.float64)

In [18]:
DATA_DIR = os.path.join("data", "DS_10283_2791", "clean_trainset_56spk_wav")

In [22]:
list_files = os.listdir(DATA_DIR)

files_len = []

for file_name in tqdm(list_files):
    signal, sr = torchaudio.load(os.path.join(DATA_DIR, file_name))
    files_len.append(signal.shape[-1] / sr)

100%|██████████| 23075/23075 [00:16<00:00, 1413.81it/s]


In [23]:
print("Average length of files in seconds:", np.mean(files_len))
print("Min length of files in seconds:", np.min(files_len))
print("Max length of files in seconds:", np.max(files_len))

Average length of files in seconds: 2.9690520368364033
Min length of files in seconds: 1.1734375
Max length of files in seconds: 16.246875
